In [1]:
import os
from pathlib import Path
import sys

if (Path(os.getcwd()) != Path("c:/Users/alexi/studcess/api")):
    os.chdir("c:/Users/alexi/studcess/api")

print(os.getcwd())

# PDF libraries
from pypdf import PdfReader

# Docling
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Regex
import re

# OpenRouter
from openrouter import OpenRouter
from openrouter.operations import CreateEmbeddingsResponse

# env
from dotenv import load_dotenv

# Calculate token size
from pytokencalc import count_tokens

# Hash
import hashlib

# Import from project files
print(sys.path)

from app.config import open_router_client
from app.config import supabase_client


c:\Users\alexi\studcess\api


c:\Users\alexi\studcess\api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['C:\\Users\\alexi\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\python314.zip', 'C:\\Users\\alexi\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\DLLs', 'C:\\Users\\alexi\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none\\Lib', 'C:\\Users\\alexi\\AppData\\Roaming\\uv\\python\\cpython-3.14-windows-x86_64-none', 'c:\\Users\\alexi\\studcess\\api\\.venv', '', 'c:\\Users\\alexi\\studcess\\api\\.venv\\Lib\\site-packages', 'c:\\Users\\alexi\\studcess\\api\\.venv\\Lib\\site-packages\\win32', 'c:\\Users\\alexi\\studcess\\api\\.venv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\alexi\\studcess\\api\\.venv\\Lib\\site-packages\\pythonwin']


# 0 - define the sample docs path

In [ ]:
docs = Path("sample-data")
sample = docs / "exam1-2022-sol-1.pdf"

# 1 - extract document metadata

In [3]:
def extract_metadata(pdf_path: Path)->None:
    reader = PdfReader(pdf_path)
    metadata = reader.metadata
    num_pages = len(reader.pages)
    
    print(metadata)
    print(num_pages)
    
    # Build custom metadata
    info = {num_pages:num_pages}
    
    # Regexes for file class 
    EXAM_KEYWORDS = ["Exam"]
    
extract_metadata(sample)

{'/Producer': 'MiKTeX GPL Ghostscript 9.25', '/CreationDate': "D:20240108104408+01'00'", '/ModDate': "D:20240108104408+01'00'", '/Creator': 'dvips(k) 2020.1 Copyright 2020 Radical Eye Software', '/Title': 'cours_ana1.dvi'}
14


# 2 - convert document to markdown with docling

In [4]:
%%time

pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_page_images=False,
    do_picture_description=False,
    ocr_options=RapidOcrOptions(),
    #picture_description_options=smolvlm_picture_description
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"

def process_document(pdf_path: Path,converter: DocumentConverter, n_pages=-1)->str:
    result = converter.convert(pdf_path)
    doc = result.document
    print(f"\nNumber of pages: {len(doc.pages)}\n")

    converted_dir = docs / "converted"
    converted_dir.mkdir(parents=True, exist_ok=True)
    
    doc.save_as_markdown(converted_dir / f"{pdf_path.stem}.md")
    doc.save_as_json(converted_dir / f"{pdf_path.stem}.json")
    
    markdown = doc.export_to_markdown(page_break_placeholder=PAGE_BREAK_PLACEHOLDER)
    
    if n_pages == -1:
        return markdown
    
    return PAGE_BREAK_PLACEHOLDER.join(markdown.split(PAGE_BREAK_PLACEHOLDER)[:n_pages])
    
doc = process_document(sample,converter,2)
print(doc)

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2329.79it/s]



Number of pages: 14

## Analyse 1

## Table des mati`eres

|   1 L'addition et la multiplication | 1 L'addition et la multiplication   | 1 L'addition et la multiplication                                                        | 3                                                                             |
|-------------------------------------|-------------------------------------|------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
|                                     | nombres                             | nombres                                                                                  |                                                                               |
|                                   2 | Les ensembles de                    |                                                                                          | 4                       

# 3 - Implement chunking 

We do a first pass by splitting by heading as this will be where we are most likely to get relevant information

In [5]:
HEADING_PATTERN = "\n#"

raw_chunks = doc.split(HEADING_PATTERN)

chunked_text = ""
for i,chunk in enumerate(raw_chunks):
    if chunk.startswith("#"):
        chunk = f"#{chunk}" # Put back the header
    chunked_text += f"<start_chunk_{i}>\n{chunk}<end_chunk_{i}>"

print(chunked_text)

<start_chunk_0>
### Analyse 1
<end_chunk_0><start_chunk_1>
## Table des mati`eres

|   1 L'addition et la multiplication | 1 L'addition et la multiplication   | 1 L'addition et la multiplication                                                        | 3                                                                             |
|-------------------------------------|-------------------------------------|------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
|                                     | nombres                             | nombres                                                                                  |                                                                               |
|                                   2 | Les ensembles de                    |                                                                                          | 4

# 4 - Embed chunks

## Golden rule for chunk sizes
We divide the length of text by 4 to get the approximate token count:
- Between 100 and 1000 => acceptable
- Less than 100 => too low, merge with another chunk or check for junk and discard
- More than 1000 => too big, split into smaller chunks needed

In [6]:
EMBEDDING_MODEL = "openai/text-embedding-3-small"

class Chunk:
    def __init__(self,embedding:list[float],text:str)->None:
        self.embedding = embedding
        self.text = text
        self.content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
        self.row = {
            "content": self.text,
            "content_hash": self.content_hash,
            "embedding": self.embedding
        }
    
    def __repr__(self)->str:
        return f"{self.row}"

def embed_texts(raw_chunks: list[str], embedding_model: str) -> list[Chunk]:
    chunks: list[Chunk] = []

    for text in raw_chunks:
        tokens_count = len(text) // 4
        print(f"tokens count {tokens_count}")

        try:
            res = open_router_client.embeddings.generate(
                input=text,
                model=embedding_model,
            )
            print("Chunk successfully embedded")
        except Exception as e:
            print(f"Error embedding chunk: {e}")

        embedding = res.data[0].embedding   # list of lists of floats
        chunks.append(Chunk(embedding, text))

    return chunks

chunks = embed_texts(raw_chunks,EMBEDDING_MODEL)

tokens count 3
Chunk successfully embedded
tokens count 2865
Chunk successfully embedded


# Store chunks inside the database

In [7]:
def store_chunks(chunks:list[Chunk]):
    for chunk in chunks:
        try:
            # Only insert new row if the content is unique, otherwise update
            response = supabase_client.table("chunks").upsert(chunk.row, on_conflict="content_hash").execute()
            print(f"Successfully inserted {len(response.data)} chunks.")
        except Exception as e:
            print(f"Error inserting data: {e}")

store_chunks(chunks)

Successfully inserted 1 chunks.
Successfully inserted 1 chunks.


# Retrieval

In [8]:
def retrieve(question: str, k: int = 3):
    q_vec = embed_texts([question],EMBEDDING_MODEL)[0]  
     
    # same model as the chunks
    try: 
        res = supabase_client.rpc("match_chunks", {
            "query_embedding": q_vec.embedding,
            "match_count": k,
        }).execute()
        print("Question embedded successfully")
        print(res.data)
    except Exception as e:
        print(f"Error embedding question: {e}")
        
    return res.data
    
retrieve("What is the solution to (b) inside Question 1?")

tokens count 11
Chunk successfully embedded
Question embedded successfully
[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352', 'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 

[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352',
  'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 2 val r: String = StringUtils.charAt(s, StringUtils.length(s)-2) + 3 "" +

# Ask the questions

In [26]:
RAG_PROMPT = """
You are a computer science professor and you are grading an exam about the Scala fundamentals.

Use the context:
{context}

To answer the question:
{question}
""".strip()

In [27]:
LLM = "openai/gpt-4o-mini"

def prompt(question: str)->str:
    chunks = retrieve(question, k=1)
    
    context = ""
    for i,chunk in enumerate(chunks):
        context += f"<Chunk {i}>"
        context += chunk['content']
        context += f"<Chunk {i}>"
    
    rag_prompt = RAG_PROMPT.format(context=context,question=question)
    
    try:
        response = open_router_client.chat.send(
            model=LLM,
            messages=[{
                "content": f"{rag_prompt}",
                "role": "system"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error prompting model: {e}")
        
answer = prompt("What is the solution to (b) inside Question 1?")
print(answer)

tokens count 11
Chunk successfully embedded
Question embedded successfully
[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352', 'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 

In [28]:
prompt(input("Write prompt:"))

tokens count 12
Chunk successfully embedded
Question embedded successfully
[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352', 'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 

'The instruction for part (c) inside Question 1 asks for the value of `b` in the expression:\n\n```scala\nval a : Boolean = ... \nval b : Boolean = a || !a\n```\n\nThe value of `b` is determined by the logical expression `a || !a`. Because `a` can only be `true` or `false`, the expression `a || !a` will always evaluate to `true`. This is known as the law of excluded middle in logic, which states that for any proposition, either it is true or its negation is true.\n\nSo, regardless of the value of `a`, `b` will always be `true`.\n\nThus, the answer for part (c) is:\n- **Valeur de b**: `true`'